In [2]:
from google.colab import drive
drive.mount('/content/drive')

!pip install lightgbm -q

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, roc_auc_score, accuracy_score
from scipy.stats import poisson
import joblib
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
print("✅ Ready")

Mounted at /content/drive
✅ Ready


In [3]:
matches = pd.read_csv('/content/drive/MyDrive/Matches.csv')
elo = pd.read_csv('/content/drive/MyDrive/EloRatings.csv')

print("Matches shape:", matches.shape)
print("ELO shape:", elo.shape)

# Preview columns
print("\nMatches columns (first 20):")
print(matches.columns.tolist()[:20])

Matches shape: (230557, 48)
ELO shape: (245033, 4)

Matches columns (first 20):
['Division', 'MatchDate', 'MatchTime', 'HomeTeam', 'AwayTeam', 'HomeElo', 'AwayElo', 'Form3Home', 'Form5Home', 'Form3Away', 'Form5Away', 'FTHome', 'FTAway', 'FTResult', 'HTHome', 'HTAway', 'HTResult', 'HomeShots', 'AwayShots', 'HomeTarget']


In [5]:
# Ensure date columns are datetime
matches['MatchDate'] = pd.to_datetime(matches['MatchDate'])
elo['date'] = pd.to_datetime(elo['date']) # Corrected 'Date' to 'date'

# Sort matches by date (important for rolling features later)
matches = matches.sort_values('MatchDate')

# Select relevant columns from ELO for merging
elo_for_merge = elo[['club', 'date', 'elo']]

# Merge home Elo
matches = matches.merge(elo_for_merge, left_on=['HomeTeam', 'MatchDate'], right_on=['club', 'date'], how='left')
# Rename the merged 'elo' column to 'HomeElo_merged' to avoid confusion with existing 'HomeElo'
matches = matches.rename(columns={'elo': 'HomeElo_merged'})
# Update the 'HomeElo' column in 'matches'
# Prioritize merged Elo, otherwise keep existing
matches['HomeElo'] = matches['HomeElo_merged'].fillna(matches['HomeElo'])
# Drop the temporary columns from the merge
matches = matches.drop(columns=['club', 'date', 'HomeElo_merged'])


# Merge away Elo
matches = matches.merge(elo_for_merge, left_on=['AwayTeam', 'MatchDate'], right_on=['club', 'date'], how='left')
matches = matches.rename(columns={'elo': 'AwayElo_merged'})
matches['AwayElo'] = matches['AwayElo_merged'].fillna(matches['AwayElo'])
matches = matches.drop(columns=['club', 'date', 'AwayElo_merged'])


# Fill missing Elo with median (for teams not in ELO table)
matches['HomeElo'] = matches['HomeElo'].fillna(matches['HomeElo'].median())
matches['AwayElo'] = matches['AwayElo'].fillna(matches['AwayElo'].median())

print("After merge:", matches.shape)


After merge: (230557, 48)


In [7]:
# Total goals (regression target)
matches['TotalGoals'] = matches['FTHome'] + matches['FTAway']

# Over/Under thresholds – we'll derive these later from predicted TotalGoals,
# but we create them here for evaluation if needed
thresholds = [0.5, 1.5, 2.5, 3.5, 4.5]
for t in thresholds:
    matches[f'Over_{t}'] = (matches['TotalGoals'] > t).astype(int)
    matches[f'Under_{t}'] = (matches['TotalGoals'] < t).astype(int)

# 1X (Home win or draw)
matches['1X'] = ((matches['FTResult'] == 'H') | (matches['FTResult'] == 'D')).astype(int)

# X2 (Draw or away win)
matches['X2'] = ((matches['FTResult'] == 'D') | (matches['FTResult'] == 'A')).astype(int)

# 12 (Home or away win, no draw)
matches['12'] = (matches['FTResult'] != 'D').astype(int)

# Clean sheets
matches['HomeCleanSheet'] = (matches['FTAway'] == 0).astype(int)
matches['AwayCleanSheet'] = (matches['FTHome'] == 0).astype(int)

# BTTS (Both Teams To Score)
matches['BTTS'] = ((matches['FTHome'] > 0) & (matches['FTAway'] > 0)).astype(int)

print("All targets created. Sample:")
print(matches[['TotalGoals', '1X', 'X2', '12', 'HomeCleanSheet', 'AwayCleanSheet', 'BTTS']].head())

All targets created. Sample:
   TotalGoals  1X  X2  12  HomeCleanSheet  AwayCleanSheet  BTTS
0         4.0   1   0   1               0               0     1
1         4.0   1   0   1               0               0     1
2         1.0   0   1   1               0               1     0
3         5.0   1   0   1               0               0     1
4         6.0   1   1   0               0               0     1


In [9]:
# List of potential feature columns – adjust based on actual column names
basic_features = [
    'HomeShots', 'AwayShots', 'HomeTarget', 'AwayTarget',
    'HomeCorners', 'AwayCorners', 'HomeFouls', 'AwayFouls',
    'HomeYellow', 'AwayYellow', 'HomeRed', 'AwayRed',
    'Form3Home', 'Form5Home', 'Form3Away', 'Form5Away',
    'HomeElo', 'AwayElo'
]

# Keep only those that exist
existing_features = [col for col in basic_features if col in matches.columns]
print("Existing basic features:", existing_features)

# --- Optimized rolling stats calculation ---
def add_optimized_rolling_team_stats(df, window=5):
    # Create a 'long' format DataFrame for team-wise performance
    home_df = df[['MatchDate', 'HomeTeam', 'FTHome', 'FTAway']].copy()
    home_df.rename(columns={'HomeTeam': 'Team', 'FTHome': 'GoalsScored', 'FTAway': 'GoalsConceded'}, inplace=True)
    home_df['IsHome'] = 1

    away_df = df[['MatchDate', 'AwayTeam', 'FTAway', 'FTHome']].copy()
    away_df.rename(columns={'AwayTeam': 'Team', 'FTAway': 'GoalsScored', 'FTHome': 'GoalsConceded'}, inplace=True)
    away_df['IsHome'] = 0

    team_performance = pd.concat([home_df, away_df], ignore_index=True)
    team_performance.sort_values(by=['Team', 'MatchDate'], inplace=True)

    # Calculate rolling averages, shifted to prevent data leakage (use stats *before* current match)
    # min_periods=1 means it will calculate even if fewer than 'window' matches are available
    team_performance['TeamGoalsScoredLast5'] = team_performance.groupby('Team')['GoalsScored'].transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).mean())
    team_performance['TeamGoalsConcededLast5'] = team_performance.groupby('Team')['GoalsConceded'].transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).mean())

    # Merge these back to the original matches DataFrame
    # Need to distinguish between home and away team stats
    home_stats = team_performance[team_performance['IsHome'] == 1][['MatchDate', 'Team', 'TeamGoalsScoredLast5', 'TeamGoalsConcededLast5']]
    home_stats.rename(columns={'Team': 'HomeTeam',
                               'TeamGoalsScoredLast5': 'HomeGoalsScoredLast5',
                               'TeamGoalsConcededLast5': 'HomeGoalsConcededLast5'}, inplace=True)

    away_stats = team_performance[team_performance['IsHome'] == 0][['MatchDate', 'Team', 'TeamGoalsScoredLast5', 'TeamGoalsConcededLast5']]
    away_stats.rename(columns={'Team': 'AwayTeam',
                               'TeamGoalsScoredLast5': 'AwayGoalsScoredLast5',
                               'TeamGoalsConcededLast5': 'AwayGoalsConcededLast5'}, inplace=True)

    # Merge home stats
    df = df.merge(home_stats, on=['MatchDate', 'HomeTeam'], how='left')

    # Merge away stats
    df = df.merge(away_stats, on=['MatchDate', 'AwayTeam'], how='left')

    return df

# Call the optimized function
matches = add_optimized_rolling_team_stats(matches, window=5)

# Fill any NaN (first few matches of a team) with a reasonable default
# Using half of the global average total goals as a fallback for goals scored/conceded.
mean_total_goals_per_match = matches['TotalGoals'].mean()
for col in ['HomeGoalsScoredLast5', 'HomeGoalsConcededLast5', 'AwayGoalsScoredLast5', 'AwayGoalsConcededLast5']:
    matches[col] = matches[col].fillna(mean_total_goals_per_match / 2)

# Final feature list
feature_cols = existing_features + [
    'HomeGoalsScoredLast5', 'HomeGoalsConcededLast5',
    'AwayGoalsScoredLast5', 'AwayGoalsConcededLast5'
]
feature_cols = [col for col in feature_cols if col in matches.columns]
print("Final feature columns:", feature_cols)


Existing basic features: ['HomeShots', 'AwayShots', 'HomeTarget', 'AwayTarget', 'HomeCorners', 'AwayCorners', 'HomeFouls', 'AwayFouls', 'HomeYellow', 'AwayYellow', 'HomeRed', 'AwayRed', 'Form3Home', 'Form5Home', 'Form3Away', 'Form5Away', 'HomeElo', 'AwayElo']
Final feature columns: ['HomeShots', 'AwayShots', 'HomeTarget', 'AwayTarget', 'HomeCorners', 'AwayCorners', 'HomeFouls', 'AwayFouls', 'HomeYellow', 'AwayYellow', 'HomeRed', 'AwayRed', 'Form3Home', 'Form5Home', 'Form3Away', 'Form5Away', 'HomeElo', 'AwayElo', 'HomeGoalsScoredLast5', 'HomeGoalsConcededLast5', 'AwayGoalsScoredLast5', 'AwayGoalsConcededLast5']


In [10]:
split_date = '2023-01-01'
train_df = matches[matches['MatchDate'] < split_date]
test_df = matches[matches['MatchDate'] >= split_date]

print(f"Train: {len(train_df)} matches, Test: {len(test_df)} matches")

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]

# Fill any missing values with training median
X_train = X_train.fillna(X_train.median())
X_test = X_test.fillna(X_train.median())

Train: 203270 matches, Test: 27287 matches


In [11]:
y_train_goals = train_df['TotalGoals']
y_test_goals = test_df['TotalGoals']

params_reg = {
    'objective': 'regression',
    'metric': 'mae',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'random_state': 42
}

model_goals = lgb.LGBMRegressor(**params_reg)
model_goals.fit(X_train, y_train_goals,
                eval_set=[(X_test, y_test_goals)],
                eval_metric='mae',
                callbacks=[lgb.early_stopping(20), lgb.log_evaluation(50)])

y_pred_goals = model_goals.predict(X_test)
mae = mean_absolute_error(y_test_goals, y_pred_goals)
print(f"Total Goals MAE: {mae:.3f}")

Training until validation scores don't improve for 20 rounds
[50]	valid_0's l1: 1.19334
[100]	valid_0's l1: 1.18062
Did not meet early stopping. Best iteration is:
[100]	valid_0's l1: 1.18062
Total Goals MAE: 1.181


In [ ]:
binary_targets = {
    '1X': train_df['1X'],
    'X2': train_df['X2'],
    '12': train_df['12'],
    'HomeCleanSheet': train_df['HomeCleanSheet'],
    'AwayCleanSheet': train_df['AwayCleanSheet'],
    'BTTS': train_df['BTTS']
}

params_bin = {
    'objective': 'binary',
    'metric': 'auc',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'random_state': 42
}

models_binary = {}
for name, y_train in binary_targets.items():
    print(f"Training {name}...")
    model = lgb.LGBMClassifier(**params_bin)
    model.fit(X_train, y_train,
              eval_set=[(X_test, test_df[name])],
              eval_metric='auc',
              callbacks=[lgb.early_stopping(20), lgb.log_evaluation(50)])
    models_binary[name] = model
    y_pred_prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(test_df[name], y_pred_prob)
    print(f"{name} AUC: {auc:.4f}\n")

Training 1X...
Training until validation scores don't improve for 20 rounds
[50]	valid_0's auc: 0.75393
[100]	valid_0's auc: 0.76205
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.76205
1X AUC: 0.7621

Training X2...
Training until validation scores don't improve for 20 rounds
[50]	valid_0's auc: 0.742723
[100]	valid_0's auc: 0.752611
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.752611
X2 AUC: 0.7526

Training 12...
Training until validation scores don't improve for 20 rounds
[50]	valid_0's auc: 0.600995
[100]	valid_0's auc: 0.602213
Did not meet early stopping. Best iteration is:
[93]	valid_0's auc: 0.602388
12 AUC: 0.6024

Training HomeCleanSheet...
Training until validation scores don't improve for 20 rounds
[50]	valid_0's auc: 0.726469
[100]	valid_0's auc: 0.730141
Did not meet early stopping. Best iteration is:
[100]	valid_0's auc: 0.730141
HomeCleanSheet AUC: 0.7301

Training AwayCleanSheet...
Training until validation scores don'